In [1]:
# --- Import Libraries ---
import random
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

In [2]:
# --- Game Logic ---
def getDigits(num):
    return [int(i) for i in str(num)]

def noDuplicates(num):
    return len(getDigits(num)) == len(set(getDigits(num)))

def generateNum():
    while True:
        num = random.randint(100, 999)
        if noDuplicates(num):
            return num

def numOfBullsCows(num, guess):
    bulls, cows = 0, 0
    num_li, guess_li = getDigits(num), getDigits(guess)
    for i, g in enumerate(guess_li):
        if g == num_li[i]:
            bulls += 1
        elif g in num_li:
            cows += 1
    return bulls, cows


In [3]:
# --- Game Variables ---
secret_num = generateNum()
tries_left = 10
history = []  # store all guesses + results

# --- UI Elements ---
game_title = widgets.HTML("<h2 style='text-align:center;color:#2e8b57;font-size:38px;'>🐮🐂 Cows and Bulls Game</h2>")
instruction_label = widgets.HTML("<b>Enter your 3-digit guess (no repeated digits):</b>")
guess_input = widgets.BoundedIntText(value=123, min=100, max=999, description='Your Guess:')
check_button = widgets.Button(description="Check Guess", button_style='info')
play_again_button = widgets.Button(description="Play Again", button_style='success')
play_again_button.layout.visibility = 'hidden'  # hidden until game ends
output_area = widgets.Output()

In [4]:
# --- Image URLs (you can replace with local paths or online links) ---
bull_img = "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcTGOqtK48UTq7BNNsS5XhUJkNVV5fEMnMH6_VWUXy1aTw&s=10"
cow_img = "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcR_S69ps6dSSBnLGopWFuxOC5siOI_ejawMoEq_-4bJqQ&s=10"
win_img = "https://img.magnific.com/free-vector/winner-gold-medal-with-green-ribbon-isolated-retro_1150-40633.jpg?semt=ais_hybrid&w=740&q=80"
lose_img = "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcRE9l0_HITvTzfDiswnUuVozEwiu0VD3NAcJFOiuaVzaQ&s=10"

In [5]:
# --- Validation Function ---
def validate_guess(guess):
    if not (100 <= guess <= 999) or not noDuplicates(guess):
        display(HTML("<p style='color:red;'>❌ Invalid guess: Enter a 3-digit number with unique digits.</p>"))
        return False
    return True


In [6]:
# --- Feedback Function ---
def show_feedback(bulls, cows, tries_left):
        display(HTML(f"""
        <div style='flex:1;text-align:center;'>
            <img src='{bull_img}' width='165'> <b style='font-size:24px;color:#00008B;'>{bulls} Bulls 🐂 </b>&nbsp;&nbsp;
            <img src='{cow_img}' width='230'> <b style='font-size:24px;color:#8B0000;'>{cows} Cows 🐮 </b>
            <p style='color:#4682b4;font-size:28px;'>Tries left: {tries_left}</p>
        </div>
        """))

In [7]:
# --- Main Game Function ---
def check_guess(button):
    global tries_left, secret_num
    with output_area:
        clear_output()
        guess = guess_input.value

        # Step 1: Validate
        if not validate_guess(guess):
            return

        # Step 2: Calculate bulls/cows
        bulls, cows = numOfBullsCows(secret_num, guess)
        tries_left -= 1

        # Step 3: Show feedback
        show_feedback(bulls, cows, tries_left)


        # Win/Lose conditions
        if bulls == 3:
            display(HTML(f"""
            <div style='text-align:center;'>
                <img src='{win_img}' width='250'>
                <h3 style='color:green;font-size:28px;'>🎉 You guessed it right!</h3>
            </div>
            """))
            guess_input.disabled = True
            check_button.disabled = True
            play_again_button.layout.visibility = 'visible'   # show button

        elif tries_left == 0:
            display(HTML(f"""
            <div style='text-align:center;'>
                <img src='{lose_img}' width='250'>
                <h3 style='color:red;font-size:28px;'>💀 Game Over! The number was {secret_num}</h3>
            </div>
            """))
            guess_input.disabled = True
            check_button.disabled = True
            play_again_button.layout.visibility = 'visible'   # show button

        # Save guess history
        history.append((guess, bulls, cows, tries_left))

        # Show history table
        table_html = "<h3 style='text-align:center;'>📜 Guess History</h3><table style='margin:auto;border-collapse:collapse;'>"
        table_html += "<tr style='background:#f0f8ff;'><th style='padding:8px;border:1px solid #ccc;'>Guess</th><th style='padding:8px;border:1px solid #ccc;'>Bulls</th><th style='padding:8px;border:1px solid #ccc;'>Cows</th><th style='padding:8px;border:1px solid #ccc;'>Tries Left</th></tr>"
        for g, b, c, t in history:
            table_html += f"<tr><td style='padding:8px;border:1px solid #ccc;'>{g}</td><td style='padding:8px;border:1px solid #ccc;'>{b}</td><td style='padding:8px;border:1px solid #ccc;'>{c}</td><td style='padding:8px;border:1px solid #ccc;'>{t}</td></tr>"
        table_html += "</table>"
        display(HTML(table_html))


In [8]:
# --- Reset Function ---
def reset_game(button):
    global secret_num, tries_left, history
    secret_num = generateNum()
    tries_left = 10
    history = []
    guess_input.disabled = False
    check_button.disabled = False
    play_again_button.layout.visibility = 'hidden'
    with output_area:
        clear_output()
        display(HTML("<p style='color:#4682b4;'>🔢 Guess the 3-digit number! You have 10 tries.</p>"))


In [9]:
# --- Event Binding ---
check_button.on_click(check_guess)
play_again_button.on_click(reset_game)

# --- Display UI ---
display(game_title, instruction_label, guess_input, check_button, play_again_button, output_area)

with output_area:
    display(HTML("<p style='color:#4682b4;'>🔢 Guess the 3-digit number! You have 10 tries.</p>"))


HTML(value="<h2 style='text-align:center;color:#2e8b57;font-size:38px;'>🐮🐂 Cows and Bulls Game</h2>")

HTML(value='<b>Enter your 3-digit guess (no repeated digits):</b>')

BoundedIntText(value=123, description='Your Guess:', max=999, min=100)

Button(button_style='info', description='Check Guess', style=ButtonStyle())

Button(button_style='success', description='Play Again', layout=Layout(visibility='hidden'), style=ButtonStyle…

Output()